# Actividad 02 — Semana 03: JOINs en SQL

**Semana:** 03  
**Tema:** JOINs en Spark SQL — INNER, LEFT, FULL OUTER, subqueries  
**Estudiante:** Daniel Guzmán  
**Notebook:** sql_joins_daniel  

## Objetivo

Practicar JOINs usando SQL sobre las tablas Delta construidas en Semana 02.

En esta actividad se trabaja con tablas Bronze sufijadas con el nombre del estudiante:

- `workspace.bronze.transactions_daniel`
- `workspace.bronze.users_daniel`
- `workspace.bronze.cards_daniel`
- `workspace.bronze.mcc_codes_daniel`
- `workspace.bronze.fraud_labels_daniel`

Se usa el sufijo `_daniel` para evitar conflictos en el catálogo compartido.

In [0]:
USE CATALOG workspace;

SHOW TABLES IN bronze;

In [0]:
SHOW TABLES IN silver;

In [0]:
DESCRIBE TABLE bronze.transactions_daniel;

In [0]:
DESCRIBE TABLE bronze.users_daniel;

In [0]:
DESCRIBE TABLE bronze.cards_daniel;

In [0]:
DESCRIBE TABLE bronze.fraud_labels_daniel;

In [0]:
SELECT COUNT(DISTINCT id) AS usuarios_unicos
FROM bronze.users_daniel;

In [0]:
SELECT
    client_id,
    COUNT(*) AS num_transacciones
FROM bronze.transactions_daniel
GROUP BY client_id
ORDER BY num_transacciones DESC
LIMIT 10;

In [0]:
SELECT COUNT(DISTINCT t.client_id) AS usuarios_en_tx_sin_perfil
FROM bronze.transactions_daniel t
LEFT JOIN bronze.users_daniel u
    ON CAST(t.client_id AS INT) = CAST(u.id AS INT)
WHERE u.id IS NULL;

## Parte 1 — Revisión del modelo de datos

Se revisó el modelo de datos desde SQL usando las tablas Bronze.

La relación principal entre transacciones y usuarios es:

`bronze.transactions_daniel.client_id → bronze.users_daniel.id`

También se validó si existen usuarios en transacciones que no tengan perfil en la tabla de usuarios. Este análisis ayuda a detectar problemas de integridad referencial antes de hacer JOINs.

## Parte 1 — Revisión del modelo de datos

Se revisó el modelo de datos desde SQL usando las tablas Bronze.

La relación principal entre transacciones y usuarios es:

`bronze.transactions_daniel.client_id → bronze.users_daniel.id`

También se validó si existen usuarios en transacciones que no tengan perfil en la tabla de usuarios. Este análisis ayuda a detectar problemas de integridad referencial antes de hacer JOINs.

### Resultado

Se encontraron **0 usuarios huérfanos**.

Esto indica que todos los `client_id` presentes en `bronze.transactions_daniel` tienen correspondencia en `bronze.users_daniel.id`.

El impacto es positivo: los JOINs entre transacciones y usuarios no deberían perder registros por falta de perfil de usuario.

In [0]:
SELECT
    t.id AS transaction_id,
    t.date AS transaction_date,
    t.amount,
    t.merchant_id,
    u.id AS user_id,
    u.birth_year,
    u.gender
FROM bronze.transactions_daniel t
INNER JOIN bronze.users_daniel u
    ON CAST(t.client_id AS INT) = CAST(u.id AS INT)
LIMIT 20;

In [0]:
SELECT 'total_transactions' AS tabla, COUNT(*) AS filas
FROM bronze.transactions_daniel

UNION ALL

SELECT 'inner_join_users' AS tabla, COUNT(*) AS filas
FROM bronze.transactions_daniel t
INNER JOIN bronze.users_daniel u
    ON CAST(t.client_id AS INT) = CAST(u.id AS INT);

## Parte 2 — INNER JOIN

El `INNER JOIN` entre `transactions` y `users` conserva únicamente las transacciones cuyo `client_id` existe en la tabla de usuarios.

Como en la validación previa se encontraron **0 usuarios huérfanos**, no se espera pérdida de registros al hacer el INNER JOIN.

Este JOIN es adecuado cuando solo queremos analizar transacciones con usuario válido registrado.

### Resultado

El total original de transacciones fue **13,305,915** y el resultado del `INNER JOIN` con usuarios también fue **13,305,915**.

No se perdieron registros porque no existen usuarios huérfanos entre `transactions.client_id` y `users.id`.

In [0]:
SELECT
    t.id AS transaction_id,
    t.amount,
    t.merchant_id,
    c.card_type,
    c.credit_limit,
    CASE WHEN c.id IS NULL THEN 'sin_tarjeta' ELSE 'con_tarjeta' END AS estado_tarjeta
FROM bronze.transactions_daniel t
LEFT JOIN bronze.cards_daniel c
    ON CAST(t.card_id AS INT) = CAST(c.id AS INT)
LIMIT 20;

In [0]:
SELECT
    CASE WHEN c.id IS NULL THEN 'sin_tarjeta' ELSE 'con_tarjeta' END AS estado,
    COUNT(*) AS total
FROM bronze.transactions_daniel t
LEFT JOIN bronze.cards_daniel c
    ON CAST(t.card_id AS INT) = CAST(c.id AS INT)
GROUP BY 1;

## Parte 3 — LEFT JOIN

El `LEFT JOIN` conserva todas las transacciones de la tabla izquierda (`transactions`), incluso si no existe una tarjeta asociada en `cards`.

Este tipo de JOIN es útil para auditoría de calidad de datos, porque permite detectar transacciones con `card_id` que no tienen correspondencia en la tabla de tarjetas sin perder las transacciones originales.

Si aparecen registros como `sin_tarjeta`, eso indicaría un problema de integridad referencial entre `transactions.card_id` y `cards.id`.
### Resultado

Todas las transacciones quedaron como `con_tarjeta`.

- `con_tarjeta`: **13,305,915**
- `sin_tarjeta`: **0**

Esto indica que no se encontraron transacciones con `card_id` huérfano. La relación entre `transactions.card_id` y `cards.id` mantiene integridad referencial.

In [0]:
WITH fraud_clean AS (
    SELECT
        id AS transaction_id,
        target AS is_fraud_label,
        CASE
            WHEN target = 'Yes' THEN 1
            WHEN target = 'No' THEN 0
            ELSE NULL
        END AS is_fraud
    FROM bronze.fraud_labels_daniel
),
mcc_ref AS (
    SELECT DISTINCT
        CAST(mcc AS INT) AS mcc,
        merchant_category
    FROM silver.transactions_daniel
)
SELECT
    t.id AS transaction_id,
    CAST(t.date AS TIMESTAMP) AS transaction_date,
    CAST(REGEXP_REPLACE(t.amount, '[$,]', '') AS DOUBLE) AS amount,
    ABS(CAST(REGEXP_REPLACE(t.amount, '[$,]', '') AS DOUBLE)) AS amount_abs,
    t.merchant_id,
    t.merchant_city,
    t.merchant_state,
    CAST(t.mcc AS INT) AS mcc,
    m.merchant_category,
    t.use_chip,
    CAST(t.client_id AS INT) AS user_id,
    u.birth_year,
    u.gender,
    u.per_capita_income,
    CAST(t.card_id AS INT) AS card_id,
    c.card_type,
    c.credit_limit,
    f.is_fraud_label,
    f.is_fraud,
    HOUR(CAST(t.date AS TIMESTAMP)) AS hora,
    DAYOFWEEK(CAST(t.date AS TIMESTAMP)) AS dia_semana,
    MONTH(CAST(t.date AS TIMESTAMP)) AS mes,
    YEAR(CAST(t.date AS TIMESTAMP)) AS anio,
    CASE
        WHEN DAYOFWEEK(CAST(t.date AS TIMESTAMP)) IN (1, 7) THEN 1
        ELSE 0
    END AS es_fin_de_semana
FROM bronze.transactions_daniel t
LEFT JOIN bronze.users_daniel u
    ON CAST(t.client_id AS INT) = CAST(u.id AS INT)
LEFT JOIN bronze.cards_daniel c
    ON CAST(t.card_id AS INT) = CAST(c.id AS INT)
LEFT JOIN mcc_ref m
    ON CAST(t.mcc AS INT) = m.mcc
LEFT JOIN fraud_clean f
    ON CAST(t.id AS BIGINT) = CAST(f.transaction_id AS BIGINT)
LIMIT 100;

## Nota técnica sobre MCC en SQL

La tabla `bronze.mcc_codes_daniel` viene en formato ancho: una fila y muchas columnas, donde cada columna representa un código MCC.

Al intentar convertirla manualmente con `STACK`, apareció un error porque uno de los códigos listados no existía exactamente como columna en la tabla Bronze.

Para mantener el foco de la actividad en JOINs SQL, se usó una referencia de MCC desde `silver.transactions_daniel`, que ya contiene la relación `mcc → merchant_category`. Los demás JOINs principales se reconstruyeron desde Bronze:

- `transactions → users`
- `transactions → cards`
- `transactions → fraud_labels`

En un pipeline productivo, lo ideal sería normalizar `mcc_codes` desde Bronze a una tabla larga antes de consumirla en SQL.

In [0]:
SELECT
    t.id AS transaction_id,
    t.amount,
    t.merchant_id,
    u.per_capita_income
FROM bronze.transactions_daniel t
INNER JOIN bronze.users_daniel u
    ON CAST(t.client_id AS INT) = CAST(u.id AS INT)
WHERE CAST(REGEXP_REPLACE(u.per_capita_income, '[$,]', '') AS DOUBLE) > (
    SELECT AVG(CAST(REGEXP_REPLACE(per_capita_income, '[$,]', '') AS DOUBLE))
    FROM bronze.users_daniel
)
ORDER BY CAST(REGEXP_REPLACE(u.per_capita_income, '[$,]', '') AS DOUBLE) DESC
LIMIT 20;

In [0]:
SELECT *
FROM (
    SELECT
        client_id,
        COUNT(*) AS num_transacciones,
        ROUND(SUM(CAST(REGEXP_REPLACE(amount, '[$,]', '') AS DOUBLE)), 2) AS gasto_total
    FROM bronze.transactions_daniel
    GROUP BY client_id
) resumen_usuario
WHERE num_transacciones > 50
ORDER BY gasto_total DESC
LIMIT 10;

In [0]:
SELECT DISTINCT
    u.id,
    u.birth_year,
    u.gender
FROM bronze.users_daniel u
WHERE EXISTS (
    SELECT 1
    FROM bronze.transactions_daniel t
    INNER JOIN bronze.fraud_labels_daniel f
        ON CAST(t.id AS BIGINT) = CAST(f.id AS BIGINT)
    WHERE CAST(t.client_id AS INT) = CAST(u.id AS INT)
      AND f.target = 'Yes'
)
LIMIT 20;

## Parte 5 — Subqueries

Se implementaron tres tipos de subqueries:

- Subquery en `WHERE`: permite comparar cada usuario contra el promedio general de ingreso per cápita.
- Subquery en `FROM`: permite crear una tabla derivada con métricas por usuario y luego filtrarla.
- `EXISTS`: permite validar si existe al menos una transacción fraudulenta asociada a cada usuario.

Estas consultas son útiles cuando se necesita construir una lógica intermedia sin crear una tabla física.

## Parte 6 — Comparación PySpark vs SQL

### Consulta elegida

**Opción A:** Usuarios con más de 5 transacciones fraudulentas, ordenados por tasa de fraude.

El objetivo es comparar cómo se escribe la misma lógica usando SQL y usando PySpark.

In [0]:
SELECT
    t.client_id AS user_id,
    COUNT(*) AS total_transacciones,
    SUM(CASE WHEN f.target = 'Yes' THEN 1 ELSE 0 END) AS total_fraudes,
    ROUND(
        SUM(CASE WHEN f.target = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) * 100,
        4
    ) AS tasa_fraude_pct
FROM bronze.transactions_daniel t
LEFT JOIN bronze.fraud_labels_daniel f
    ON CAST(t.id AS BIGINT) = CAST(f.id AS BIGINT)
GROUP BY t.client_id
HAVING SUM(CASE WHEN f.target = 'Yes' THEN 1 ELSE 0 END) > 5
ORDER BY tasa_fraude_pct DESC, total_fraudes DESC
LIMIT 20;

In [0]:

%python
from pyspark.sql import functions as F

df_tx = spark.table("workspace.bronze.transactions_daniel")
df_fraud = spark.table("workspace.bronze.fraud_labels_daniel")

df_resultado_pyspark = (
    df_tx.alias("t")
    .join(
        df_fraud.alias("f"),
        F.col("t.id").cast("long") == F.col("f.id").cast("long"),
        "left"
    )
    .groupBy(F.col("t.client_id").alias("user_id"))
    .agg(
        F.count("*").alias("total_transacciones"),
        F.sum(F.when(F.col("f.target") == "Yes", 1).otherwise(0)).alias("total_fraudes")
    )
    .withColumn(
        "tasa_fraude_pct",
        F.round(F.col("total_fraudes") / F.col("total_transacciones") * 100, 4)
    )
    .filter(F.col("total_fraudes") > 5)
    .orderBy(F.col("tasa_fraude_pct").desc(), F.col("total_fraudes").desc())
)

display(df_resultado_pyspark.limit(20))

## Reflexión — PySpark vs SQL

Para esta consulta, SQL resulta más legible porque la lógica es principalmente analítica: hacer un JOIN, agrupar, calcular métricas y ordenar resultados.

SQL es más fácil de leer para usuarios analíticos y de negocio, especialmente cuando la pregunta se expresa como una consulta directa sobre tablas.

PySpark sería más conveniente si esta lógica hiciera parte de una pipeline más grande, con varias transformaciones reutilizables, validaciones, escritura de tablas o manejo programático de errores.

Si el requerimiento cambia ligeramente, por ejemplo cambiar el umbral de fraudes o agregar una columna, SQL es más rápido de modificar. Si el cambio implica transformar múltiples DataFrames o construir una capa de datos persistente, PySpark sería más mantenible.